# Heart Disease — clean and preprocess

Goal: prepare `heart.csv` so a later model can learn real clinical signal, not data-entry artifacts.

Every step below says **what** we did and **why**. This notebook stops after preprocessing. It does **not** train a model.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set(style='whitegrid')
pd.set_option('display.max_columns', 50)

df = pd.read_csv('heart.csv')
df.head()


## 1. Inspect the raw file

**Why:** you cannot clean what you have not measured. We check shape, types, nulls, duplicates, and impossible numeric values before touching anything.

In [2]:
print('shape:', df.shape)
print('\ncolumn types:')
print(df.dtypes)
print('\nnull counts:')
print(df.isnull().sum())
print('\nduplicate rows:', df.duplicated().sum())
print('\ntarget balance:')
print(df['HeartDisease'].value_counts())
print(df['HeartDisease'].value_counts(normalize=True).round(3))
df.describe()

shape: (918, 12)

column types:
Age                 int64
Sex                object
ChestPainType      object
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG         object
MaxHR               int64
ExerciseAngina     object
Oldpeak           float64
ST_Slope           object
HeartDisease        int64
dtype: object

null counts:
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

duplicate rows: 0

target balance:
HeartDisease
1    508
0    410
Name: count, dtype: int64
HeartDisease
1    0.553
0    0.447
Name: proportion, dtype: float64


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


### Findings from inspect

| check | result | decision |
|---|---|---|
| pandas nulls | 0 | no `dropna` — missingness is hidden as zeros |
| duplicates | 0 | nothing to drop |
| target | 55% disease / 45% no disease | **no SMOTE**. Stratified split is enough. Oversampling a nearly balanced set just risks overfitting |
| `RestingBP` min | 0 | blood pressure of 0 is not compatible with life → treat as missing |
| `Cholesterol` min | 0 | serum cholesterol of 0 is not compatible with life → treat as missing |
| `Oldpeak` min | −2.6 | possible ST elevation vs rest; keep (explained below) |

## 2. How each input relates to disease

**Why:** this tells us which columns are signal, which encodings to use, and which "odd" values are actually informative.

In [3]:
cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope', 'FastingBS']

for col in cat_cols:
    print(f'\n===== {col} vs HeartDisease (row % ) =====')
    print(pd.crosstab(df[col], df['HeartDisease'], normalize='index').round(3))

print('\nnumeric correlation with HeartDisease:')
print(df.corr(numeric_only=True)['HeartDisease'].sort_values(key=abs, ascending=False).round(3))


===== Sex vs HeartDisease (row % ) =====
HeartDisease      0      1
Sex                       
F             0.741  0.259
M             0.368  0.632

===== ChestPainType vs HeartDisease (row % ) =====
HeartDisease       0      1
ChestPainType              
ASY            0.210  0.790
ATA            0.861  0.139
NAP            0.645  0.355
TA             0.565  0.435

===== RestingECG vs HeartDisease (row % ) =====
HeartDisease      0      1
RestingECG                
LVH           0.436  0.564
Normal        0.484  0.516
ST            0.343  0.657

===== ExerciseAngina vs HeartDisease (row % ) =====
HeartDisease        0      1
ExerciseAngina              
N               0.649  0.351
Y               0.148  0.852

===== ST_Slope vs HeartDisease (row % ) =====
HeartDisease      0      1
ST_Slope                  
Down          0.222  0.778
Flat          0.172  0.828
Up            0.803  0.197

===== FastingBS vs HeartDisease (row % ) =====
HeartDisease      0      1
FastingBS           

### What the rates tell us (keep these features)

- **ST_Slope:** Flat ~83% disease, Down ~78%, Up ~20%. Strongest categorical signal. One-hot encode, do not drop this column.
- **ChestPainType:** ASY ~79% disease, ATA ~14%. Nominal (not ordered), so one-hot, not 0/1/2/3 labels. Label-encoding would fake an order the model might trust.
- **ExerciseAngina Y:** ~85% disease. Simple binary map.
- **Sex:** men ~63% disease, women ~26%. Binary map. We do **not** drop it even though the classes are uneven — that is biology, not a data bug.
- **FastingBS=1:** ~79% disease. Already 0/1, leave as-is.
- **RestingECG:** weaker (Normal ~52%, ST ~66%). Still keep; one-hot. Weak is not the same as useless.

## 3. Hidden missing values: Cholesterol = 0 and RestingBP = 0

**Why this is the most important cleaning step.**  
172 rows (19%) have Cholesterol 0. A living person does not have cholesterol 0. Those zeros are **unmeasured** labs, common in this Kaggle set.

If we leave them as 0:
- the numeric correlation of Cholesterol vs disease becomes **negative** (because zeros cluster in the disease class)
- that is the opposite of clinical reality (higher cholesterol raises risk)
- the model learns a fake rule: "cholesterol 0 → disease"

So we do two things:
1. **Flag** `CholesterolMissing` — missingness itself predicts disease (~88% of zero-cholesterol rows have disease). Throwing that away would lose signal.
2. **Replace 0 with NaN, then impute the median of real values** so the cholesterol number the model sees is a plausible lab, not a fake zero.

RestingBP = 0 appears in **one** row. Same logic, no extra flag (one row cannot teach a missingness pattern).

In [4]:
print('Cholesterol == 0:', (df['Cholesterol'] == 0).sum())
print('RestingBP == 0:', (df['RestingBP'] == 0).sum())
print('\nHeartDisease rate when Cholesterol == 0:')
print(pd.crosstab(df['Cholesterol'] == 0, df['HeartDisease'], normalize='index').round(3))

print('\nCholesterol stats when it is actually measured:')
print(df.loc[df['Cholesterol'] > 0, 'Cholesterol'].describe())

Cholesterol == 0: 172
RestingBP == 0: 1

HeartDisease rate when Cholesterol == 0:
HeartDisease      0      1
Cholesterol               
False         0.523  0.477
True          0.116  0.884

Cholesterol stats when it is actually measured:
count    746.000000
mean     244.635389
std       59.153524
min       85.000000
25%      207.250000
50%      237.000000
75%      275.000000
max      603.000000
Name: Cholesterol, dtype: float64


## 4. Oldpeak negatives and high cholesterol outliers

**Oldpeak < 0 (13 rows):** Oldpeak is ST depression vs rest. A negative value can mean ST elevation. We **keep** them. Clipping to 0 would invent a measurement we did not observe, and 13 rows will not dominate the fit.

**Cholesterol > 400 (13 rows):** possible severe hypercholesterolemia. Rare but real. We **keep** them. Winsorizing would hide a high-risk tail the model should see.

**Why we do not delete rows:** 918 is already small. Dropping 172 zero-cholesterol rows would throw away 19% of the data, mostly disease cases, and bias the model toward "no disease".

## 5. Apply cleaning on a copy

Work on `data` so `df` stays the raw file for comparison.

In [5]:
data = df.copy()

# Flag first, while 0 still means "not measured".
data['CholesterolMissing'] = (data['Cholesterol'] == 0).astype(int)

# Impossible zeros → NaN. Impute later using TRAIN medians only (no leakage).
data['Cholesterol'] = data['Cholesterol'].replace(0, np.nan)
data['RestingBP'] = data['RestingBP'].replace(0, np.nan)

print('NaNs after replacing impossible zeros:')
print(data[['Cholesterol', 'RestingBP']].isna().sum())
data.head()

NaNs after replacing impossible zeros:
Cholesterol    172
RestingBP        1
dtype: int64


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,CholesterolMissing
0,40,M,ATA,140.0,289.0,0,Normal,172,N,0.0,Up,0,0
1,49,F,NAP,160.0,180.0,0,Normal,156,N,1.0,Flat,1,0
2,37,M,ATA,130.0,283.0,0,ST,98,N,0.0,Up,0,0
3,48,F,ASY,138.0,214.0,0,Normal,108,Y,1.5,Flat,1,0
4,54,M,NAP,150.0,195.0,0,Normal,122,N,0.0,Up,0,0


## 6. Encode categoricals

| column | method | why |
|---|---|---|
| `Sex` | M=1, F=0 | two levels, no extra dummy needed |
| `ExerciseAngina` | Y=1, N=0 | two levels, yes/no |
| `ChestPainType` | one-hot | 4 unordered types. Integers 0–3 would tell the model TA > NAP > ATA > ASY, which is false |
| `RestingECG` | one-hot | 3 unordered types |
| `ST_Slope` | one-hot | 3 unordered types, very strong predictor |
| `FastingBS` | already 0/1 | leave |

`drop_first=False` so tree models can split on every level (including ASY and Flat). Linear models can handle the extra column at this size.

In [6]:
data['Sex'] = data['Sex'].map({'M': 1, 'F': 0})
data['ExerciseAngina'] = data['ExerciseAngina'].map({'Y': 1, 'N': 0})

data = pd.get_dummies(
    data,
    columns=['ChestPainType', 'RestingECG', 'ST_Slope'],
    dtype=int,
)

print(data.columns.tolist())
data.head()

['Age', 'Sex', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'HeartDisease', 'CholesterolMissing', 'ChestPainType_ASY', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST', 'ST_Slope_Down', 'ST_Slope_Flat', 'ST_Slope_Up']


,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,CholesterolMissing,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,1,140.0,289.0,0,172,0,0.0,0,0,0,1,0,0,0,1,0,0,0,1
1,49,0,160.0,180.0,0,156,0,1.0,1,0,0,0,1,0,0,1,0,0,1,0
2,37,1,130.0,283.0,0,98,0,0.0,0,0,0,1,0,0,0,0,1,0,0,1
3,48,0,138.0,214.0,0,108,1,1.5,1,0,1,0,0,0,0,1,0,0,1,0
4,54,1,150.0,195.0,0,122,0,0.0,0,0,0,0,1,0,0,1,0,0,0,1


## 7. One extra feature: age-adjusted heart rate

`MaxHR` falls with age, so a MaxHR of 140 is high for a 70-year-old and low for a 30-year-old.

`hr_ratio = MaxHR / (220 - Age)` is the usual clinical "percent of age-predicted max".  
**Why add it:** it lets the model use exercise capacity relative to age instead of treating MaxHR as an absolute number. We **keep** raw `Age` and `MaxHR` as well — the ratio does not replace them.

In [7]:
data['hr_ratio'] = data['MaxHR'] / (220 - data['Age'])
data.head()

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,CholesterolMissing,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up,hr_ratio
0,40,1,140.0,289.0,0,172,0,0.0,0,0,0,1,0,0,0,1,0,0,0,1,0.955556
1,49,0,160.0,180.0,0,156,0,1.0,1,0,0,0,1,0,0,1,0,0,1,0,0.912281
2,37,1,130.0,283.0,0,98,0,0.0,0,0,0,1,0,0,0,0,1,0,0,1,0.535519
3,48,0,138.0,214.0,0,108,1,1.5,1,0,1,0,0,0,0,1,0,0,1,0,0.627907
4,54,1,150.0,195.0,0,122,0,0.0,0,0,0,0,1,0,0,1,0,0,0,1,0.734940


## 8. Train/test split **before** impute and scale

**Why this order matters:** if you impute or scale on the full 918 rows, test-set numbers leak into the medians and standard deviations. The accuracy you would report would be slightly fake.

- 80/20 split
- `stratify=y` so both sides keep ~55% disease
- `random_state=42` so the split is repeatable

In [8]:
X = data.drop(columns=['HeartDisease'])
y = data['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print('train:', X_train.shape, 'test:', X_test.shape)
print('train disease rate:', y_train.mean().round(3))
print('test disease rate:', y_test.mean().round(3))

train: (734, 20) test: (184, 20)
train disease rate: 0.553
test disease rate: 0.554


## 9. Impute + scale (fit on train only)

**Impute median** for `Cholesterol` and `RestingBP`:
- median is robust to the 400–600 cholesterol tail
- we must **not** use the mean of a column that still contained zeros — those zeros are already NaN

**StandardScaler** on continuous columns only:
- needed for logistic regression (otherwise Age and Cholesterol dominate the loss)
- trees ignore scale, so it does not hurt Random Forest
- **do not scale** 0/1 dummies — scaling them makes coefficients harder to read and does not help trees

We store `X_train_model` / `X_test_model` for training.

In [9]:
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'hr_ratio']
binary_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols),
        ('bin', SimpleImputer(strategy='most_frequent'), binary_cols),
    ]
)

X_train_model = preprocess.fit_transform(X_train)
X_test_model = preprocess.transform(X_test)

feature_names = numeric_cols + binary_cols
X_train_model = pd.DataFrame(X_train_model, columns=feature_names, index=X_train.index)
X_test_model = pd.DataFrame(X_test_model, columns=feature_names, index=X_test.index)

print('NaNs after preprocess:', X_train_model.isna().sum().sum(), X_test_model.isna().sum().sum())
X_train_model.head()

NaNs after preprocess: 0 0


,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,hr_ratio,Sex,FastingBS,ExerciseAngina,CholesterolMissing,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
485,0.970012,0.339016,-0.517499,-0.324520,0.317046,-0.032663,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
486,0.122028,-1.266031,-0.572835,1.689837,-0.440356,1.880455,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
117,0.546020,-0.159102,1.714383,-0.247045,0.601071,-0.087034,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
361,-0.725956,1.501291,-0.130147,-0.479470,-0.819056,-0.716523,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
296,-0.407962,0.671094,-0.130147,0.101594,-0.156330,-0.016279,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


## Decision log (short)

1. **Did not drop nulls/duplicates** — none in the file.
2. **Did not SMOTE** — 55/45 is close enough; a stratified split is enough.
3. **Cholesterol 0 → missing flag + median impute** — zeros are unmeasured labs and would teach a fake negative cholesterol–disease link.
4. **RestingBP 0 → median impute** — one impossible row.
5. **Kept negative Oldpeak and high cholesterol** — rare but clinically possible; 918 rows is too small to throw away tails.
6. **One-hot for ChestPain / ECG / ST slope** — categories are not ordered.
7. **Binary map Sex and ExerciseAngina** — two levels.
8. **Added `hr_ratio`** — MaxHR depends on age.
9. **Split, then impute/scale** — no test leakage.
10. **Scale continuous only** — dummies stay 0/1.

Preprocessing stops here. Use `X_train_model`, `X_test_model`, `y_train`, `y_test` if you train a model later.
